# Layer 5 — Inferensi pada Satu Struk Maret

Notebook ini memuat `models/xgboost_cross_sell_ril.pkl` dan menilai satu baris sungguhan dari data uji Maret.
Ambang keputusannya 0,50. Keluaran adalah probabilitas kelas 1 dan keputusan ditawarkan atau tidak.

Delapan fitur harus berurutan sama dengan saat pelatihan.


In [ ]:
from pathlib import Path

import joblib
import pandas as pd

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "models" / "xgboost_cross_sell_ril.pkl").exists():
            return candidate
    raise FileNotFoundError("Jalankan Layer 4 terlebih dahulu.")

PROJECT_DIR = find_project_dir()
MODEL_PATH = PROJECT_DIR / "models" / "xgboost_cross_sell_ril.pkl"
TEST_PATH = PROJECT_DIR / "outputs_ril" / "layer3_test_features.csv"
FEATURE_COLUMNS = [
    "current_basket_size", "hour_of_day", "day_of_week", "is_weekend",
    "rule_confidence", "rule_lift", "antecedent_rate", "interest_lift",
]
THRESHOLD = 0.50

print("--> [INFO] Memuat model data riil dan satu baris uji Maret...")
model = joblib.load(MODEL_PATH)
sample = pd.read_csv(TEST_PATH, dtype={"NO_BKT": str, "antecedent": str, "consequent": str}).iloc[[0]]
features = sample[FEATURE_COLUMNS]
probability = float(model.predict_proba(features)[0, 1])
decision = "Tawarkan" if probability >= THRESHOLD else "Tidak ditawarkan"
print("Struk", sample["NO_BKT"].iloc[0])
print("Antecedent", sample["antecedent"].iloc[0], "-> consequent", sample["consequent"].iloc[0])
print("Label aktual", int(sample["y"].iloc[0]))
for name in FEATURE_COLUMNS:
    print(f"  {name:<22} {float(features[name].iloc[0]):.6f}")
print(f"Probabilitas kelas 1 : {probability:.6f}")
print(f"Keputusan            : {decision}")
